# Evolving Art 🎨
Recreate a photo with **50 semi-transparent circles**, found by pure trial and error: change one circle at random, keep the change if the picture got closer to the target. No gradients, just a (1+1) evolution strategy (hill climbing). Watch an abstract painting emerge.

Background: [[Gradient Descent]] (and what to do when you have no gradient) · overview: [[Fun Projects]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_sample_image
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
H = W = 64
target = load_sample_image("flower.jpg")[::6, ::6][:H, 20:20 + W].astype(float) / 255     # bundled with scikit-learn, cropped to 64×64
YY, XX = np.mgrid[0:H, 0:W]
loss = lambda img: float(np.mean((img - target) ** 2))
N_CIRCLES = 50
def random_circle(rng): return [rng.uniform(0, W), rng.uniform(0, H), rng.uniform(2, 15), *rng.random(3), rng.uniform(0.1, 0.6)]
def mutate(circles, rng):
    new = [c[:] for c in circles]; i = rng.integers(len(new))
    if rng.random() < 0.1: new[i] = random_circle(rng)                                            # replace a circle
    else:
        j = rng.integers(7); scale = [4, 4, 2, .1, .1, .1, .05][j]
        new[i][j] = float(np.clip(new[i][j] + rng.normal(0, scale), [0, 0, 1, 0, 0, 0, 0.05][j], [W, H, 30, 1, 1, 1, 0.9][j]))
    return new
plt.imshow(target); plt.axis("off"); plt.title("target"); plt.show()

## 1. Render
Start from a white canvas and paint each circle `(x, y, r, red, green, blue, alpha)` in order: inside the circle, `canvas = (1 − alpha)·canvas + alpha·colour`.

In [ ]:
def render(circles):
    # TODO 1: alpha-blend each filled circle onto a white H×W×3 canvas
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_render():
    img = render([[10, 10, 3, 1, 0, 0, 1.0], [10, 10, 3, 0, 0, 1, 0.5]])
    return img.shape == (H, W, 3) and np.allclose(img[10, 10], [0.5, 0, 0.5]) and np.allclose(img[30, 30], 1) and np.allclose(img[10, 13], [0.5, 0, 0.5]) and np.allclose(img[10, 14], 1)
check("render", _t_render, "Red then half-transparent blue on the same spot → (0.5, 0, 0.5); outside stays white; the radius is inclusive.")

## 2. Evolve
(1+1)-ES: repeat `iters` times: `mutate` the current circles, render, and **keep the mutant if its loss is ≤ the current loss**. Return `(circles, loss_history)` where the history has the best loss after every iteration (plus the start).

In [ ]:
def evolve(circles, iters=4000, seed=0):
    # TODO 2: hill climbing with mutate/render/loss
    raise NotImplementedError  # TODO 2

In [ ]:
if ready("render"):
    try:
        rng = np.random.default_rng(0); start = [random_circle(rng) for _ in range(N_CIRCLES)]
        art, hist = evolve(start)
        fig, ax = plt.subplots(1, 3, figsize=(11, 3.6))
        ax[0].imshow(render(start)); ax[0].set_title(f"start, loss {hist[0]:.3f}"); ax[1].imshow(render(art)); ax[1].set_title(f"after {len(hist) - 1} steps, loss {hist[-1]:.3f}")
        ax[2].imshow(target); ax[2].set_title("target")
        for a in ax: a.axis("off")
        plt.show(); plt.plot(hist); plt.xlabel("iteration"); plt.ylabel("MSE"); plt.show()
        check("art improves", lambda: hist[-1] < 0.5 * hist[0] and all(np.diff(hist) <= 1e-12), "The loss should more than halve and never increase (only accept improvements).")
    except NotImplementedError:
        print("❌ art improves: not implemented yet – fill in the TODO above.")

<details><summary>▶ Solution</summary>

```python
def render(circles):
    canvas = np.ones((H, W, 3))
    for x, y, r, red, green, blue, a in circles:
        mask = (XX - x) ** 2 + (YY - y) ** 2 <= r ** 2
        canvas[mask] = (1 - a) * canvas[mask] + a * np.array([red, green, blue])
    return canvas
```

```python
def evolve(circles, iters=4000, seed=0):
    rng = np.random.default_rng(seed)
    best_loss = loss(render(circles)); hist = [best_loss]
    for it in range(iters):
        cand = mutate(circles, rng)
        l = loss(render(cand))
        if l <= best_loss:
            circles, best_loss = cand, l
        hist.append(best_loss)
    return circles, hist
```
</details>

## Stretch goals
- Use your own photo (resize to 64×64), or a portrait in triangles instead of circles.
- Simulated annealing: sometimes accept a worse picture early on. Does it escape local optima?
- Make the renderer differentiable (soft circle edges) and use **gradient descent** with PyTorch. Faster? Better?